# Cross-Lingual GraphRAG Pipeline on Kaggle
Ensure you have set the Accelerator to **GPU T4 x2** before running this notebook.
This notebook runs the complete pipeline with **Neo4j Graph Integration** and **Persian Translation**.

In [ ]:
import os
import sys
# Ensure we are in the root working directory and remove any existing repo
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/MedRAG

# Clone the MedRAG repository from GitHub
!git clone https://github.com/TeleEng/MedRAG.git

# Change working directory into the repo
os.chdir('MedRAG')

print("=== Repository Version Info ===")
!git log -1 --format="Commit: %h | Date: %cd"
print("===============================")

# Install dependencies
!pip install -q -r requirements.txt

# Add repo root to Python path so 'from src...' imports work
if '.' not in sys.path:
    sys.path.insert(0, '.')


### Step 0: Neo4j Secrets Configuration
Load Neo4j credentials from Kaggle Secrets so they are available to the pipeline.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
try:
    os.environ["NEO4J_URI"] = user_secrets.get_secret("NEO4J_URI")
    os.environ["NEO4J_USERNAME"] = user_secrets.get_secret("NEO4J_USERNAME")
    os.environ["NEO4J_PASSWORD"] = user_secrets.get_secret("NEO4J_PASSWORD")
    print("Neo4j Secrets Loaded Successfully!")
except Exception as e:
    print("Warning: Please configure NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD in Kaggle Secrets.")

### Step 1: Data Preparation
Download the `medalpaca` dataset and process it into chunks.

In [ ]:
from src.data_prep import load_and_prepare_data
load_and_prepare_data()

### Step 2: Indexing (FAISS, BM25, and Neo4j Graph)
Build the Dense/Sparse indexes and push entities to Neo4j AuraDB.

In [ ]:
from src.indexer import build_indexes
from src.graph_indexer import GraphIndexer

# 1. Local Indexes
build_indexes()

# 2. Graph Database Indexing
g_indexer = GraphIndexer()
g_indexer.build_graph()
g_indexer.close()

### Step 3: QLoRA Fine-Tuning
Instruction-tune the base model.

In [ ]:
from src.trainer import train_model
train_model()

### Step 4: End-to-End Cross-Lingual Generation
Run the full pipeline. The system will translate Persian -> English, query Neo4j+FAISS, generate in English, and translate back to Persian.

In [ ]:
from src.generator import MedRAGPipeline

pipeline = MedRAGPipeline()

query_pes = "علائم بیماری دیابت چیست؟"
res_pes, res_eng, docs = pipeline.answer_query(query_pes)

print("\n==========================")
print("MEDRAG PERSIAN ANSWER:")
print("==========================")
print(res_pes)

print("\n==========================")
print("ORIGINAL ENGLISH ANSWER:")
print("==========================")
print(res_eng)